<a href="https://colab.research.google.com/github/omerlutfiduran/senior-design-project/blob/main/Senior_Desing_Project_Data_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import ee
ee.Authenticate()

In [3]:
ee.Initialize(project='senior-design-project-489117')


In [4]:
import ee
import folium

# Antalya sınırlarını tanımlama (FAO GAUL Level 1 veri setini kullanıyoruz)
antalya_fc = ee.FeatureCollection("FAO/GAUL/2015/level1") \
               .filter(ee.Filter.eq('ADM1_NAME', 'Antalya'))

# NASA FIRMS Aktif Yangın Veri Setini Çağırma
# Test için büyük Manavgat yangınlarının olduğu 2021 tarihini seçiyoruz.
start_date = '2021-07-01'
end_date = '2021-08-31'

firms_dataset = ee.ImageCollection('FIRMS') \
                  .filterBounds(antalya_fc) \
                  .filterDate(start_date, end_date)

# Sadece ısı anormalliği olan pikselleri (T21 bandı) seçip Antalya sınırına kırpıyoruz
fires = firms_dataset.select('T21').max().clip(antalya_fc)

# Harita Üzerinde Görselleştirme
# GEE görüntülerini Folium haritasına eklemek için yardımcı fonksiyon
def add_ee_layer(self, ee_image_object, vis_params, name):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

# Antalya merkezli interaktif bir harita oluşturalım
my_map = folium.Map(location=[36.9, 30.7], zoom_start=8)

# Antalya sınırını haritaya ekleme
empty = ee.Image().byte()
outline = empty.paint(featureCollection=antalya_fc, color=1, width=2)
my_map.add_ee_layer(outline, {'palette': ['000000']}, 'Antalya Siniri')

# Yangın noktalarını haritaya ekleme (Isı şiddetine göre renk gradyanı)
# 300 Kelvin (sarı) ile 500 Kelvin (koyu kırmızı) arasında doğrusal bir renk geçişi sağlanır.
fire_vis_params = {
    'min': 300,
    'max': 500,
    'palette': ['yellow', 'orange', 'red', 'darkred']
}
my_map.add_ee_layer(fires, fire_vis_params, '2021 Yanginlari (FIRMS - Isi Siddeti)')

# Harita katman kontrolünü ekle ve göster
my_map.add_child(folium.LayerControl())
display(my_map)

In [5]:
import pandas as pd
import ee

# 1. Yangın Olan Noktaları (1) Çekme
# Haritadaki kırmızı piksellerden test amaçlı 200 tanesinin koordinatını alıyoruz
fire_samples = fires.sample(
    region=antalya_fc,
    scale=1000,       # MODIS uydusunun çözünürlüğü (1km)
    numPixels=200,    # Test aşamasında olduğumuz için şimdilik 200 satır
    geometries=True
)

def add_fire_label(feature):
    return feature.set('YANGIN_DURUMU', 1)

fire_labeled = fire_samples.map(add_fire_label)

# 2. Yangın Olmayan Noktaları (0) Oluşturma
# Antalya sınırları içine rastgele 200 nokta atıyoruz (Sağlıklı orman / Çoğunluk Sınıfı)
non_fire_samples = ee.FeatureCollection.randomPoints(region=antalya_fc, points=200)

def add_non_fire_label(feature):
    return feature.set('YANGIN_DURUMU', 0)

non_fire_labeled = non_fire_samples.map(add_non_fire_label)

# 3. İki Veri Setini Birleştirme (Toplam 400 satırlık ham verimiz)
combined_points = fire_labeled.merge(non_fire_labeled)

# 4. GEE Verisini Python Tablosuna (Pandas DataFrame) Çevirme
# Sistemden Enlem, Boylam ve Etiket (0/1) bilgilerini sıyırıyoruz
def extract_coords_and_labels(feature):
    coords = feature.geometry().coordinates()
    label = feature.get('YANGIN_DURUMU')
    return ee.Feature(None, {'Boylam': coords.get(0), 'Enlem': coords.get(1), 'YANGIN_DURUMU': label})

final_dataset_ee = combined_points.map(extract_coords_and_labels)

# Veriyi Google sunucularından bizim Colab ortamımıza indiriyoruz
data_list = final_dataset_ee.reduceColumns(ee.Reducer.toList(3), ['Enlem', 'Boylam', 'YANGIN_DURUMU']).get('list').getInfo()

# Python (Pandas) tablosu oluşturma
df = pd.DataFrame(data_list, columns=['Enlem', 'Boylam', 'YANGIN_DURUMU'])

# Tablonun ilk 5 ve son 5 satırını gözlem için ekrana bastırma
print("\nTabo oluşturuldu")
display(df.head())
display(df.tail())


Tabo oluşturuldu


,Enlem,Boylam,YANGIN_DURUMU
0,36.999414,31.367858,1
1,36.884091,31.390846,1
2,36.992086,31.340318,1
3,36.917897,31.051564,1
4,36.843344,31.537043,1


,Enlem,Boylam,YANGIN_DURUMU
206,36.977152,29.925556,0
207,36.732105,29.726226,0
208,37.296497,30.324046,0
209,37.206580,31.508505,0
210,36.903078,29.943865,0
